## Import PySpark

In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

builder = (
    SparkSession.builder
    .master("local")
    .appName("PySpark Demo")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/workspaces/endjin-pyspark-examples/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bc1129f8-9c31-4906-91cb-4ca615c536a8;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.2.0 in central
	found org.roaringbi

## Read Data into DataFrame

In [2]:
COLUMN_NAMES = [
        "id", "price", "date", "postcode", "property_type",
        "old_new", "duration", "paon", "saon", "street",
        "locality", "town_city", "district", "county",
        "ppd_category", "record_type",
    ]

In [3]:
land_registry_data = (
    spark.read
    .option("header", False)
    .option("nullValue", "")
    .csv("../../data/land_registry_data/pp-*.csv")
    .toDF(*COLUMN_NAMES)
    .withColumn("price", F.col("price").cast("long"))
)

26/04/21 16:43:29 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../../data/land_registry_data/pp-*.csv.
java.io.FileNotFoundException: File ../../data/land_registry_data/pp-*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [4]:
land_registry_data.show()

+--------------------+------+----------------+--------+-------------+-------+--------+-----------------+-------+------------------+---------------+-----------+--------------------+-----------+------------+-----------+
|                  id| price|            date|postcode|property_type|old_new|duration|             paon|   saon|            street|       locality|  town_city|            district|     county|ppd_category|record_type|
+--------------------+------+----------------+--------+-------------+-------+--------+-----------------+-------+------------------+---------------+-----------+--------------------+-----------+------------+-----------+
|{D707E535-5720-0A...|260000|2021-08-06 00:00|SO45 2HT|            T|      N|       F|               17|   NULL|   PERRYWOOD CLOSE|        HOLBURY|SOUTHAMPTON|          NEW FOREST|  HAMPSHIRE|           A|          A|
|{D707E535-5721-0A...|375000|2021-09-01 00:00|SO23 7FR|            S|      N|       F|                1|   NULL|    BOXALL GARDE

In [5]:
land_registry_data = (
    land_registry_data
    .withColumn("date", F.to_date("date", "yyyy-MM-dd HH:mm"))
)

In [6]:
land_registry_data.groupBy("property_type").count().show()

+-------------+-------+
|property_type|  count|
+-------------+-------+
|            F| 878918|
|            T|1344870|
|            D|1135907|
|            O| 264930|
|            S|1317496|
+-------------+-------+



In [7]:
land_registry_data = land_registry_data.withColumn(
    "property_type",
    F.when(F.col("property_type") == "D", "Detached")
     .when(F.col("property_type") == "S", "Semi-Detached")
     .when(F.col("property_type") == "T", "Terraced")
     .when(F.col("property_type") == "F", "Flat")
     .when(F.col("property_type") == "O", "Other")
     .otherwise(F.col("property_type")),
)

In [8]:
land_registry_data.groupBy("property_type").count().show()

+-------------+-------+
|property_type|  count|
+-------------+-------+
|Semi-Detached|1317496|
|         Flat| 878918|
|     Terraced|1344870|
|        Other| 264930|
|     Detached|1135907|
+-------------+-------+



In [9]:
land_registry_data = land_registry_data.filter(F.col("property_type") != "Other")

In [10]:
land_registry_data = (
    land_registry_data
    .withColumn("year", F.year("date"))
)

In [11]:
land_registry_data.select("id", "date", "year", "property_type", "price").show()

+--------------------+----------+----+-------------+------+
|                  id|      date|year|property_type| price|
+--------------------+----------+----+-------------+------+
|{D707E535-5720-0A...|2021-08-06|2021|     Terraced|260000|
|{D707E535-5721-0A...|2021-09-01|2021|Semi-Detached|375000|
|{D707E535-5723-0A...|2021-06-28|2021|         Flat|132000|
|{D707E535-5724-0A...|2021-09-10|2021|     Terraced|295000|
|{D707E535-5725-0A...|2021-08-27|2021|     Terraced|360000|
|{D707E535-5726-0A...|2021-06-30|2021|     Detached|490500|
|{D707E535-5729-0A...|2021-06-25|2021|     Terraced|240000|
|{D707E535-572A-0A...|2021-05-10|2021|         Flat|146000|
|{D707E535-572B-0A...|2021-01-20|2021|         Flat|215000|
|{D707E535-572D-0A...|2021-08-26|2021|     Terraced|260250|
|{D707E535-572F-0A...|2021-03-31|2021|     Terraced|300000|
|{D707E535-5730-0A...|2021-08-16|2021|     Detached|330000|
|{D707E535-5732-0A...|2021-08-25|2021|     Terraced|215000|
|{D707E535-5733-0A...|2021-09-02|2021|  

In [12]:
annual_price_by_property_type = (
    land_registry_data
    .groupBy("year", "property_type")
    .agg(
        F.expr("percentile(price, 0.5)").alias("median_price"),
    )
    .orderBy("year", "property_type")
)

In [13]:
annual_price_by_property_type.show(truncate=False)

+----+-------------+------------+
|year|property_type|median_price|
+----+-------------+------------+
|2021|Detached     |386950.0    |
|2021|Flat         |230000.0    |
|2021|Semi-Detached|244995.0    |
|2021|Terraced     |210000.0    |
|2022|Detached     |420000.0    |
|2022|Flat         |236000.0    |
|2022|Semi-Detached|260000.0    |
|2022|Terraced     |220000.0    |
|2023|Detached     |417000.0    |
|2023|Flat         |231000.0    |
|2023|Semi-Detached|260000.0    |
|2023|Terraced     |215000.0    |
|2024|Detached     |410000.0    |
|2024|Flat         |235000.0    |
|2024|Semi-Detached|262500.0    |
|2024|Terraced     |220000.0    |
|2025|Detached     |416100.5    |
|2025|Flat         |230000.0    |
|2025|Semi-Detached|270000.0    |
|2025|Terraced     |225000.0    |
+----+-------------+------------+



## Visualise the results

In [14]:
import plotly.express as px

In [15]:
px.line(
    annual_price_by_property_type.toPandas(),
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [16]:
annual_price_by_property_type.write.format("delta").mode("overwrite").save("../../data/price_paid_insights/annual_price_by_property_type")

In [17]:
# PySpark DataFrames are lazy by default — no computation happens
# until an action (.show(), .collect(), .write) is called.
# This entire chain builds a logical plan without executing anything.
lazy_result = (
    spark.read
    .option("header", False)
    .option("nullValue", "")
    .csv("../../data/land_registry_data/pp-*.csv")
    .toDF(*COLUMN_NAMES)
    .withColumn("price", F.col("price").cast("long"))
    .withColumn("date", F.to_date("date", "yyyy-MM-dd HH:mm"))
    .withColumn(
        "property_type",
        F.when(F.col("property_type") == "D", "Detached")
         .when(F.col("property_type") == "S", "Semi-Detached")
         .when(F.col("property_type") == "T", "Terraced")
         .when(F.col("property_type") == "F", "Flat")
         .when(F.col("property_type") == "O", "Other")
         .otherwise(F.col("property_type")),
    )
    .filter(F.col("property_type") != "Other")
    .withColumn("year", F.year("date"))
    .groupBy("year", "property_type")
    .agg(
        F.expr("percentile(price, 0.5)").alias("median_price"),
    )
    .orderBy("year", "property_type")
)

26/04/21 16:45:16 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../../data/land_registry_data/pp-*.csv.
java.io.FileNotFoundException: File ../../data/land_registry_data/pp-*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

Unlike Polars (which has separate eager `DataFrame` and lazy `LazyFrame` types), PySpark DataFrames are **always lazy**. No computation happens until an action like `.show()`, `.collect()`, or `.write` is called. This means you can inspect the optimised execution plan before triggering any work.

In [18]:
lazy_result.explain(True)

== Parsed Logical Plan ==
'Sort ['year ASC NULLS FIRST, 'property_type ASC NULLS FIRST], true
+- Aggregate [year#681, property_type#680], [year#681, property_type#680, percentile(price#678L, cast(0.5 as double), 1, 0, 0, false) AS median_price#682]
   +- Project [id#661, price#678L, date#679, postcode#664, property_type#680, old_new#666, duration#667, paon#668, saon#669, street#670, locality#671, town_city#672, district#673, county#674, ppd_category#675, record_type#676, year(date#679) AS year#681]
      +- Filter NOT (property_type#680 = Other)
         +- Project [id#661, price#678L, date#679, postcode#664, CASE WHEN (property_type#665 = D) THEN Detached WHEN (property_type#665 = S) THEN Semi-Detached WHEN (property_type#665 = T) THEN Terraced WHEN (property_type#665 = F) THEN Flat WHEN (property_type#665 = O) THEN Other ELSE property_type#665 END AS property_type#680, old_new#666, duration#667, paon#668, saon#669, street#670, locality#671, town_city#672, district#673, county#674, pp

In [19]:
lazy_result.show(truncate=False)

+----+-------------+------------+
|year|property_type|median_price|
+----+-------------+------------+
|2021|Detached     |386950.0    |
|2021|Flat         |230000.0    |
|2021|Semi-Detached|244995.0    |
|2021|Terraced     |210000.0    |
|2022|Detached     |420000.0    |
|2022|Flat         |236000.0    |
|2022|Semi-Detached|260000.0    |
|2022|Terraced     |220000.0    |
|2023|Detached     |417000.0    |
|2023|Flat         |231000.0    |
|2023|Semi-Detached|260000.0    |
|2023|Terraced     |215000.0    |
|2024|Detached     |410000.0    |
|2024|Flat         |235000.0    |
|2024|Semi-Detached|262500.0    |
|2024|Terraced     |220000.0    |
|2025|Detached     |416100.5    |
|2025|Flat         |230000.0    |
|2025|Semi-Detached|270000.0    |
|2025|Terraced     |225000.0    |
+----+-------------+------------+

